# Convergence sweep: mesh resolution vs S-matrix accuracy

Sweeps grid resolution and tracks how |R| and |T| converge toward the reference S-matrix.
Validates that subpixel permittivity smoothing gives monotonic convergence.

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np

import meow as mw
from meow.fde.post_process import post_process_modes

In [ ]:
# --- Geometry (same as rib_waveguide_interface notebooks) ---
wl = 1.0
widths = [0.8, 1.0]
t_slab, t_soi, t_ox = 0.2, 1.5, 0.2
n_Si, n_SiO2 = 3.4, 1.4
w_sim, h_sim, y_bot = 3.0, 2.5, -0.5

si = mw.IndexMaterial(n=n_Si, name="Si", meta={"color": "orange"})
sio2 = mw.IndexMaterial(n=n_SiO2, name="SiO2", meta={"color": "steelblue"})
env = mw.Environment(wl=wl, T=25.0)

ip = partial(mw.inner_product, symmetric=False, conjugate=True)


def make_cross_section(w, num_cells):
    mesh = mw.Mesh2D(
        x=np.linspace(-w_sim / 2, w_sim / 2, num_cells),
        y=np.linspace(y_bot, y_bot + h_sim, num_cells),
    )
    structures = [
        mw.Structure2D(
            material=sio2,
            geometry=mw.Rectangle(
                x_min=-w_sim / 2, x_max=w_sim / 2, y_min=0.0, y_max=t_ox
            ),
        ),
        mw.Structure2D(
            material=si,
            geometry=mw.Rectangle(
                x_min=-w_sim / 2, x_max=w_sim / 2, y_min=t_ox, y_max=t_ox + t_slab
            ),
        ),
        mw.Structure2D(
            material=si,
            geometry=mw.Rectangle(
                x_min=-w / 2, x_max=w / 2, y_min=t_ox, y_max=t_ox + t_soi
            ),
        ),
    ]
    return mw.CrossSection(structures=structures, mesh=mesh, env=env)


def overlap_matrix(modes_a, modes_b, ip_func):
    N = len(modes_a)
    M = len(modes_b)
    O = np.zeros((N, M), dtype=complex)
    for i in range(N):
        for j in range(M):
            O[i, j] = ip_func(modes_a[i], modes_b[j])
    return O

In [ ]:
# --- Convergence sweep ---
resolutions = [51, 101, 151, 201, 301]
num_modes = 10

results = []

for nc in resolutions:
    print(f"\n=== Resolution: {nc} x {nc} ===")
    cs_left = make_cross_section(widths[0], nc)
    cs_right = make_cross_section(widths[1], nc)

    mL = mw.compute_modes(
        cs_left,
        num_modes=num_modes,
        post_process=partial(post_process_modes, inner_product=ip),
    )
    mR = mw.compute_modes(
        cs_right,
        num_modes=num_modes,
        post_process=partial(post_process_modes, inner_product=ip),
    )
    N = min(len(mL), len(mR))
    mL, mR = mL[:N], mR[:N]
    print(f"  Modes: {N} per port")

    O_LR = overlap_matrix(mL, mR, ip)
    O_RL = overlap_matrix(mR, mL, ip)

    A_LR = O_LR + O_RL.conj().T
    T_LR = np.linalg.solve(A_LR.T, 2.0 * np.eye(N)).T
    R_LL = 0.5 * (O_RL.conj().T - O_LR) @ T_LR

    A_RL = O_RL + O_LR.conj().T
    T_RL = np.linalg.solve(A_RL.T, 2.0 * np.eye(N)).T
    R_RR = 0.5 * (O_LR.conj().T - O_RL) @ T_RL

    S = np.block([[R_LL, T_RL], [T_LR, R_RR]])

    # Passivity correction
    U, sigma, Vh = np.linalg.svd(S, full_matrices=False)
    S_p = (U * np.minimum(sigma, 1.0)) @ Vh

    results.append(
        {
            "nc": nc,
            "N": N,
            "R_LL_00": abs(S_p[0, 0]),
            "T_LR_00": abs(S_p[N, 0]),
            "R_RR_00": abs(S_p[N, N]),
            "T_RL_00": abs(S_p[0, N]),
            "R_LL_00_raw": abs(S[0, 0]),
            "T_LR_00_raw": abs(S[N, 0]),
            "cond_A": np.linalg.cond(A_LR),
            "max_sigma_raw": np.linalg.svd(S, compute_uv=False).max(),
        }
    )
    r = results[-1]
    print(f"  |R_LL[0,0]| = {r['R_LL_00']:.6f}  (raw: {r['R_LL_00_raw']:.6f})")
    print(f"  |T_LR[0,0]| = {r['T_LR_00']:.6f}  (raw: {r['T_LR_00_raw']:.6f})")
    print(f"  |R_RR[0,0]| = {r['R_RR_00']:.6f}")
    print(f"  cond(A_LR)  = {r['cond_A']:.3e}")
    print(f"  max sigma   = {r['max_sigma_raw']:.6f}")

In [ ]:
# --- Reference values ---
S_ref = np.load("assets/s_ref.npy")
R_ref = abs(S_ref[0, 0])
T_ref = abs(S_ref[2, 0])
print(f"Reference: |R_LL[0,0]| = {R_ref:.6f}, |T_LR[0,0]| = {T_ref:.6f}")

# --- Results table ---
print(
    f"\n{'nc':>6}  {'|R_LL|':>10}  {'|R_RR|':>10}  {'|T_LR|':>10}  {'|T_RL|':>10}  {'cond(A)':>12}  {'max_sigma':>10}"
)
print("-" * 80)
for r in results:
    print(
        f"{r['nc']:>6}  {r['R_LL_00']:>10.6f}  {r['R_RR_00']:>10.6f}  "
        f"{r['T_LR_00']:>10.6f}  {r['T_RL_00']:>10.6f}  "
        f"{r['cond_A']:>12.3e}  {r['max_sigma_raw']:>10.6f}"
    )

In [ ]:
# --- Convergence plots ---
ncs = [r["nc"] for r in results]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# |R| convergence
ax = axes[0]
ax.semilogy(ncs, [r["R_LL_00"] for r in results], "o-", label="|R_LL[0,0]|")
ax.semilogy(ncs, [r["R_RR_00"] for r in results], "s-", label="|R_RR[0,0]|")
ax.axhline(R_ref, color="k", ls="--", alpha=0.5, label=f"ref = {R_ref:.4f}")
ax.set_xlabel("Grid points per axis")
ax.set_ylabel("|R|")
ax.set_title("Reflection convergence")
ax.legend()
ax.grid(True, alpha=0.3)

# |T| convergence
ax = axes[1]
ax.plot(ncs, [r["T_LR_00"] for r in results], "o-", label="|T_LR[0,0]|")
ax.plot(ncs, [r["T_RL_00"] for r in results], "s-", label="|T_RL[0,0]|")
ax.axhline(T_ref, color="k", ls="--", alpha=0.5, label=f"ref = {T_ref:.4f}")
ax.set_xlabel("Grid points per axis")
ax.set_ylabel("|T|")
ax.set_title("Transmission convergence")
ax.legend()
ax.grid(True, alpha=0.3)

# Condition number
ax = axes[2]
ax.semilogy(ncs, [r["cond_A"] for r in results], "o-")
ax.set_xlabel("Grid points per axis")
ax.set_ylabel("cond(O_LR + O_RL†)")
ax.set_title("Condition number")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()